# Week13 Capstone Project -  RL MAB refinement

## Strategy
For the final round, the search is framed through a lightweight **reinforcement learning / multi-armed bandit lens**.

Each candidate micro-edit is treated like an **arm**. The policy chooses among a small set of deterministic local actions built from:
- the best recent point
- the latest point
- the second-best recent point
- the centroid of the top-performing recent points
- moves along the dominant PCA direction of the top-performing cluster

Candidates are scored with a UCB-style bandit rule:

$$\text{score}(x)=\mu(x)+\beta(d)\sigma(x)-0.10\,\text{dist}(x,\text{best})-0.04\,\text{dist}(x,\text{latest})-0.08\,\text{dist}(x,\text{centroid})+0.02\,\text{novelty}(x)$$

with

$$\beta(d)=0.06+0.02d$$

This treats the final iteration as an exploit-heavy but still uncertainty-aware policy update: stronger regions are prioritised, while a small exploration bonus remains for robustness.


In [1]:
import json
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [6]:
data_dir = Path('data')
logs_dir = data_dir / 'logs'
logs_dir.mkdir(exist_ok=True)
history = pd.read_csv(data_dir / 'weekly_results/Week13.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
80,11,1,2,0.637630,0.627596,NaN,NaN,NaN,NaN,NaN,NaN,1.634249
81,11,2,2,0.514922,0.689513,NaN,NaN,NaN,NaN,NaN,NaN,0.423790
82,11,3,3,0.368779,0.570036,0.474139,NaN,NaN,NaN,NaN,NaN,-0.012440
83,11,4,4,0.399312,0.393998,0.365716,0.443779,NaN,NaN,NaN,NaN,0.074888
84,11,5,4,0.998969,0.005479,1.000000,1.000000,NaN,NaN,NaN,NaN,4426.252405
85,11,6,5,0.533367,0.179553,0.636726,0.882496,0.029005,NaN,NaN,NaN,-0.502859
86,11,7,6,0.016927,0.178931,0.289589,0.171546,0.395200,0.668006,NaN,NaN,2.119297
87,11,8,8,0.034243,0.435164,0.009022,0.325311,0.989289,0.045049,0.099288,0.699815,9.589215


In [7]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history):
    X0, y0 = load_initial(function_id)
    rows = history[history['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, Xw, yw, d

def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=seed, n_restarts_optimizer=1)

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [8]:
cfg = {
    1: {'grid': 0.0005, 'step': 0.0010},
    2: {'grid': 0.0025, 'step': 0.0050},
    3: {'grid': 0.0005, 'step': 0.0010},
    4: {'grid': 0.0002, 'step': 0.0004},
    5: {'grid': 0.000001, 'step': 0.000001},
    6: {'grid': 0.0010, 'step': 0.0020},
    7: {'grid': 0.0002, 'step': 0.0004},
    8: {'grid': 0.0001, 'step': 0.0002},
}

def beta_by_dimension(d):
    return 0.06 + 0.02 * d

def build_rl_candidates(best_x, latest_x, second_x, centroid, direction, d, step, grid):
    cands = []
    anchors = [best_x, latest_x, second_x, centroid, (best_x + latest_x) / 2.0, (best_x + centroid) / 2.0, (2 * best_x + centroid) / 3.0]
    for a in anchors:
        cands.append(round_grid(a, grid))
    for mult in [-2, -1, -0.5, 0.5, 1, 2]:
        cands.append(round_grid(best_x + mult * step * direction, grid))
        cands.append(round_grid(centroid + mult * step * direction, grid))
    for i in range(min(3, d)):
        x = best_x.copy(); x[i] += step; cands.append(round_grid(x, grid))
        x = best_x.copy(); x[i] -= step; cands.append(round_grid(x, grid))
    return np.unique(np.array(cands), axis=0)


In [ ]:
decision_logs = []
rows_out = []
for function_id in range(1, 9):
    X, y, Xw, yw, d = load_combined(function_id, history)
    best_idx = int(np.argmax(yw))
    second_idx = int(np.argsort(yw)[-2])
    best_x = Xw[best_idx].copy()
    second_x = Xw[second_idx].copy()
    latest_x = Xw[-1].copy()
    top_idx = np.argsort(yw)[-min(5, len(yw)):]
    centroid = Xw[top_idx].mean(axis=0)
    pca = PCA(n_components=1)
    pca.fit(Xw[top_idx])
    direction = pca.components_[0]
    direction = direction / (np.linalg.norm(direction) + 1e-12)
    grid = cfg[function_id]['grid']
    step = cfg[function_id]['step']
    beta = beta_by_dimension(d)
    cands = build_rl_candidates(best_x, latest_x, second_x, centroid, direction, d, step, grid)
    dist_hist = np.sqrt(((cands[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist_hist.min(axis=1)
    novelty_floor = grid / 2 if grid < 0.001 else 0.0005
    keep = min_dist >= novelty_floor
    cands = cands[keep]
    min_dist = min_dist[keep]
    gp = build_gp(1500 + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)
    mean, std = gp.predict(cands, return_std=True)
    centroid_dist = np.sqrt(((cands - centroid) ** 2).sum(axis=1)) / np.sqrt(d)
    best_dist = np.sqrt(((cands - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((cands - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + beta * std - 0.10 * best_dist - 0.04 * latest_dist - 0.08 * centroid_dist + 0.02 * min_dist
    idx = int(np.argmax(score))
    chosen = cands[idx]
    print(f'Function {function_id}: {format_point(chosen)}')
    row = {'function': function_id, 'd': d}
    for i, v in enumerate(chosen, start=1):
        row[f'x{i}'] = round(float(v), 6)
    rows_out.append(row)
    decision_logs.append({
        'function': function_id,
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
        'centroid': format_point(centroid),
        'chosen': format_point(chosen),
        'beta': beta,
    })
week13_df = pd.DataFrame(rows_out)
week13_df.to_csv('Week13_suggestions.csv', index=False)
with open(logs_dir / 'week13_decisions.json', 'w', encoding='utf-8') as f:
    json.dump(decision_logs, f, indent=2)
week13_df
